# Tiger HLM Setup
Generate lookups, YAML, and SLURM files for GPU runoff + GPU/CPU routing runs.

Install: `pip install git+https://github.com/markwang0/tiger-hlm-tools.git`

Routing uses one GPU node and one CPU node with counter traversal.
Generated wrappers load CUDA only on the GPU rank and launch with `srun --mpi=pmi2`.
After updating the package, restart the kernel and rerun setup to regenerate scripts and wrappers.
Keep `submit=False` to generate files without submitting jobs.


In [ ]:
from tiger_hlm_setup import (
    generate_lookup,
    get_forcing_characteristics,
    setup_longterm,
    setup_forecast,
    setup_spinup,
    describe_setup_options,
)

## 1. Generate forcing lookups

In [ ]:
proj = '/scratch/gpfs/GVILLARI/ra1055/GARD_LENS'

params_csv  = f'{proj}/parameters/CONUS_East_runoff_params_updated_ix3.csv'
pr_ncfile   = f'{proj}/forcings/GARDLENS_canesm5_r10i1p1f1/forcing/pcp/GARDLENS_canesm5_r10i1p1f1_pcp_19510101_19511231.nc'
t2_ncfile   = f'{proj}/forcings/GARDLENS_canesm5_r10i1p1f1/forcing/t_mean/GARDLENS_canesm5_r10i1p1f1_t_mean_19510101_19511231.nc'
lookup_dir  = f'{proj}/lookups'

# Inspect forcing files to get variable names, dims, resolution
pr_varname, pr_dims, pr_resolution = get_forcing_characteristics(pr_ncfile)
t2_varname, t2_dims, t2_resolution = get_forcing_characteristics(t2_ncfile, varname='t_mean')
print(pr_varname, pr_dims, pr_resolution)
print(t2_varname, t2_dims, t2_resolution)

# Build lookup CSVs  (flip_dims=True for IMERG)
generate_lookup(pr_ncfile, params_csv, f'{lookup_dir}/GARDLENS_East_lookup_pr.csv')
generate_lookup(t2_ncfile, params_csv, f'{lookup_dir}/GARDLENS_East_lookup_t2m.csv')

## 2. Routing partition
Create once from the routing CSV, for example for CONUS East:

```bash
RT_CSV=/scratch/gpfs/GVILLARI/ra1055/GARD_LENS/parameters/CONUS_East_routing_params_huc4_pred.csv
PART=/scratch/gpfs/GVILLARI/ra1055/GARD_LENS/partitions/East_g1c1.part  # g1c1: one GPU rank, one CPU rank
mkdir -p "$(dirname "$PART")"  # make partitions directory if it doesn't exist
python -m tiger_hlm_setup.partition_hybrid $RT_CSV 1 $PART
```

The partition file should be remade if:
 - routing CSV changes
 - number of GPU or CPU ranks changes

In [ ]:
partition_file = f'{proj}/partitions/East_g1c1.part'
sav_path = '/scratch/gpfs/GVILLARI/HLM/CONUS_V1_0_0/sav/East_sav.csv'


## 3. Set up multiple GARDLENS members

In [ ]:
import os
import glob


In [ ]:
root = f'{proj}/forcings'
members_list = sorted([
    os.path.basename(d)
    for d in glob.glob(os.path.join(root, "GARDLENS_*"))
    if os.path.isdir(d)
])

n_members = len(members_list)

In [ ]:
members_list

In [ ]:
member_indices = range(39, 42)
for id in member_indices:
    print(members_list[id])


In [ ]:
for id in member_indices:
#for id in range(0, n_members):

    member_name = members_list[id]
    member_grp = member_name.split('_')[1]
    year_end = 2024

    if member_grp in {'canesm5','cesm2'}:
        year_start = 1994
    else:
        year_start = 1994
        
    runoff_inputs = {
        'params_dir':       f'{proj}/parameters/',
        'params_csv':       'CONUS_East_runoff_params_updated_ix3.csv',
        'forcings_dir':     f'{proj}/forcings/'+ str(member_name) +'/forcing',
        'pr_file_pattern':  'pcp/'+str(member_name)+'_pcp_{start}_{end}.nc',
        'pr_varname':       pr_varname,
        'pr_resolution':    pr_resolution,
        'pr_dims':          pr_dims,
        't2_file_pattern':  't_mean/'+str(member_name)+'_t_mean_{start}_{end}.nc',
        't2_varname':       t2_varname,
        't2_dims':          t2_dims,
        'lookup_dir':       lookup_dir,
        "chunk_days":        7,
        'lookup_pr_csv':    'GARDLENS_East_lookup_pr.csv',
        'lookup_t2m_csv':   'GARDLENS_East_lookup_t2m.csv',
    }
    
    routing_inputs = {
        'partition_file': partition_file,
        'params':    f'{proj}/parameters/CONUS_East_routing_params_huc4_pred.csv',
        'sav_path':  sav_path,
        'out_flag':  0,
        'out_level': 1
    }
    
    # Optional: override SLURM defaults
    slurm_cfg = {
        'runoff_module': 'Tiger_HLM_Runoff_OudinPET',
        'runoff_version': '1.0.0',
        'routing_version': '1.1.0',
        'routing_gpu_cpus': 12,
        'routing_cpus': 112,
        'routing_cpu_partition': 'cpu',
        'routing_mem': '500G',
        'account':  'gvillari',
        'partition':'gvillari',
        #'email':    'ra1055@princeton.edu',
        'remove_runoff': True
    }
    
    setup_longterm(
        proj_root          = f'{proj}/',
        start_year         = int(year_start),
        end_year           = int(year_end),
        runoff_inputs      = runoff_inputs,
        routing_inputs     = routing_inputs,
        region             = 'East',
        product            = str(member_name),
        runoff_spinup_file = f'{proj}/spinup/final_19931231_19931231.nc',
        slurm_cfg          = slurm_cfg,
        submit             = False,   # set True to submit
    )

## 4. Submit generated jobs
Set `submit=True` only when ready. Runoff jobs submit routing after runoff succeeds.

In [ ]:
import subprocess

def submit_job(job_script, job_dir):
    subprocess.run(["sbatch", job_script], cwd=job_dir, check=True)


In [ ]:
submit = False
year_start = 1994
year_end = 2024
member_name = 'GARDLENS_canesm5_r5i1p2f1'
type = 'runoff' # or routing

for t in range(year_start,(year_end+1)):

    slurm_name = f'{type}_{t}.slurm'
    member_path =f'{proj}/{type}/{member_name}/slurm/East/'
    
    if submit:
        submit_job(slurm_name, member_path)

## 5. Spinup
Repeat 1950 for five cycles per member. Uses the same seven-day chunks as the long-term setup.

In [ ]:
for id in range(0, n_members):

    member_name = members_list[id]

    runoff_inputs = {
        'params_dir':       f'{proj}/parameters',
        'params_csv':       'CONUS_East_runoff_params_updated_ix3.csv',
        'forcings_dir':     f'{proj}/forcings/'+ str(member_name) + '/forcing',
        'pr_file_pattern':  'pcp/' + str(member_name) + '_pcp_{start}_{end}.nc',
        'pr_varname':       pr_varname,
        'pr_resolution':    pr_resolution,
        'pr_dims':          pr_dims,
        't2_file_pattern':  't_mean/'+ str(member_name) + '_t_mean_{start}_{end}.nc',
        't2_varname':       t2_varname,
        't2_dims':          t2_dims,
        'lookup_dir':       lookup_dir,
        'chunk_days':       7,
        'lookup_pr_csv':    'GARDLENS_East_lookup_pr.csv',
        'lookup_t2m_csv':   'GARDLENS_East_lookup_t2m.csv',
    }
    
    routing_inputs = {
        'partition_file': partition_file,
        'params':    f'{proj}/parameters/CONUS_East_routing_params_huc4_pred.csv',
        'sav_path':  sav_path,
    }
    
    # Optional: override SLURM defaults
    slurm_cfg = {
        'runoff_module': 'Tiger_HLM_Runoff_OudinPET',
        'runoff_version': '1.0.0',
        'routing_version': '1.1.0',
        'routing_gpu_cpus': 12,
        'routing_cpus': 112,
        'routing_cpu_partition': 'cpu',
        'routing_mem': '500G',
        'account':  'gvillari',
        'partition':'gvillari',
        'email':    'ra1055@princeton.edu',
    }
    
    setup_spinup(
        proj_root          = f'{proj}/spinup/'+str(member_name)+'/',
        spinup_year        = 1950,
        n_cycles           = 5,
        runoff_inputs      = runoff_inputs,
        routing_inputs     = routing_inputs,
        region             = 'East',
        product            =  member_name,
        runoff_spinup_file = f'{proj}/spinup/final_19801230_19801231.nc',
        slurm_cfg          = slurm_cfg,
        submit             = False,
    )